- TO DO:
    - ~~Currently, the mlp archtecture is specialized to the MNIST classification problem. I need to generalize each learning step for any data set multiclassification problem~~
    - Mini-batches learning process
    - Stochastic gradient descent
    - Math documentation
    - Inferance examples
    - Vizualization (learning and inference process)


- Open questions:
    - How to find the best number of layers and nodes?
    - How to find the best combination of activation functions?
    - If I am working with mini-batches, is there a optimal batch size? If yes, how to computate?
    - Did I shuffle my data? Was it necessary? Justify.

In [ ]:
import numpy as np
import idx2numpy

In [8]:
# read data
X_train = idx2numpy.convert_from_file('data/mnist/train-images.idx3-ubyte')
y_train = idx2numpy.convert_from_file('data/mnist/train-labels.idx1-ubyte')

X_train = (X_train.reshape(X_train.shape[0], -1) / 255.0).T
y_train = y_train.reshape(1, -1)

print("shape:")
print("X_train -", X_train.shape)
print("y_train -", y_train.shape)

shape:
X_train - (784, 60000)
y_train - (1, 60000)


This NN is designed to support any archtecture in terms of layers and nodes. For the MNIST multiclassification problem, our input layer $a^{[0]}$ will have 784 units corresponding to the 784 pixel in each 28x28 input image. A hidden layer $a^{[1]}$ will have 256 units with ReLU activation, and finally the output layer $a^{[2]}$ will have 10 units corresponding to the ten digit classes with softmax activation.

**Forward propagation**

$$Z^{[i]} = W^{[i]}A^{[i-1]} + b^{[i]}$$

$$A^{[i]} = g_{\text{ReLU}}(Z^{[i]})$$

**Backward propagation**

$$g_{\text{fudeu}}$$

**Parameter updates**

$$W^{[i]} = W^{[i]} - alpha * dW^{[i]}$$

$$b^{[i]} = b^{[i]} - alpha * b^{[i]}$$

In [41]:
def ReLU(Z):
    return np.maximum(Z, 0)

def ReLU_deriv(Z):
    return Z > 0

def softmax(Z):
    expZ = np.exp(Z - np.max(Z, axis=0, keepdims=True))
    return expZ / np.sum(expZ, axis=0, keepdims=True)

def init_params(layer_dims):
    W, b = [], []
    for i in range(len(layer_dims) - 1):
        n_in = layer_dims[i]
        n_out = layer_dims[i + 1]
        
        w_i = np.random.randn(n_out, n_in) * np.sqrt(2.0 / n_in)
        b_i = np.zeros((n_out, 1))
        
        W.append(w_i)
        b.append(b_i)

    return W, b

def forward_prop(X, W, b):
    Z, A = [], []
    A_actual = X

    for i in range(len(W)):
        zi = W[i].dot(A_actual) + b[i]
        Z.append(zi)

        if i == len(W) - 1:
            ai = softmax(zi)
        else:
            ai = ReLU(zi)

        A.append(ai)
        A_actual = ai

    return Z, A

def one_hot_encode(y, n_labels=None):
    if n_labels is None:
        n_labels = np.max(y) + 1
    
    one_hot_y = np.zeros((y.size, n_labels))
    one_hot_y[np.arange(y.size), y] = 1
    return one_hot_y.T

def get_loss(A, one_hot_y):
    m = one_hot_y.shape[1]
    loss = - (1.0 / m) * np.sum(one_hot_y * np.log(A[-1] + 1e-8))
    return loss

def softmax_jacobian(a_vec):
    a_vec = a_vec.reshape(-1, 1)
    diag_A = np.diagflat(a_vec)
    outer_A = np.dot(a_vec, a_vec.T)
    J = diag_A - outer_A
    
    return J

def back_prop(Z, A, W, X, y):
    n_layers = len(W)
    m = X.shape[1]

    dW = [None] * n_layers
    db = [None] * n_layers

    one_hot_y = one_hot_encode(y)

    dA = - one_hot_y / (A[-1] + 1e-8)

    for i in range(n_layers - 1, -1, -1):
        A_prev = X if i == 0 else A[i - 1]

        if i == n_layers - 1:
            dZ = np.zeros_like(Z[i])

            for col in range(m):
                a_col = A[i][:, col:col+1]
                dA_col = dA[:, col:col+1]
                
                J_softmax = softmax_jacobian(a_col)
                
                dZ[:, col:col+1] = np.dot(J_softmax.T, dA_col)

        else:
            dZ = dA * ReLU_deriv(Z[i])

        dW[i] = (1.0 / m) * np.dot(dZ, A_prev.T)
        db[i] = (1.0 / m) * np.sum(dZ, axis=1, keepdims=True)

        if i > 0:
            dA = np.dot(W[i].T, dZ)

    return dW, db

def update_params(W, b, dW, db, alpha):
    for i in range(len(W)):
        W[i] -= alpha * dW[i]
        b[i] -= alpha * db[i]

    return W, b

def get_predictions(a2):
    return np.argmax(a2, axis=0)

def get_accuracy(predictions, y):
    return np.sum(predictions == y) / y.size

def mlp_train(x, y, layer_dims, iterations, alpha):
    W, b = init_params(layer_dims)

    for i in range(iterations):
        Z, A = forward_prop(x, W, b)
        dW, db = back_prop(Z, A, W, x, y)
        W, b = update_params(W, b, dW, db, alpha)

        if i % 10 == 0 or i == iterations - 1:
            predictions = get_predictions(A[-1])
            acc = get_accuracy(predictions, y)
            print(f"Iteration {i} | acc: {acc}")

    return W, b

In [ ]:
W, b = mlp_train(X_train, y_train, [784, 256, 10], 200, alpha=0.5)

Iteration 0 | acc: 0.09871666666666666
Iteration 10 | acc: 0.10218333333333333
Iteration 20 | acc: 0.11236666666666667
Iteration 30 | acc: 0.11236666666666667
Iteration 40 | acc: 0.11236666666666667
Iteration 50 | acc: 0.11236666666666667
Iteration 60 | acc: 0.11236666666666667
Iteration 70 | acc: 0.11236666666666667
Iteration 80 | acc: 0.11236666666666667
Iteration 90 | acc: 0.11236666666666667
Iteration 100 | acc: 0.11236666666666667
Iteration 110 | acc: 0.11236666666666667
Iteration 120 | acc: 0.11236666666666667
Iteration 130 | acc: 0.11236666666666667
Iteration 140 | acc: 0.11236666666666667
Iteration 150 | acc: 0.11236666666666667
Iteration 160 | acc: 0.11236666666666667
Iteration 170 | acc: 0.11236666666666667
Iteration 180 | acc: 0.11236666666666667
Iteration 190 | acc: 0.11236666666666667
Iteration 199 | acc: 0.11236666666666667
